# 04 — RAG evaluation and release gates

## Scenario: should Northstar Cloud ship a new retriever?

A query-rewrite and reranking configuration looks promising in a demo. Before it reaches customers, evaluate retrieval, citation support, abstention, latency, cost, and difficult slices. This self-contained notebook uses deterministic fixtures: it teaches the evaluation contract before introducing an LLM judge or vendor platform.

## Evaluation map

```text
versioned cases -> pipeline trace -> retrieval metrics
                                  |
                                  +-> answer/citation/abstention metrics
                                  +-> latency/cost/safety checks
                                                |
                                                v
                                      explicit ship / no-ship gate
```

A high average is insufficient: a tenant-isolation, no-answer, or high-impact slice can be a release blocker.

## 1. Build a golden dataset with slices

Each case supplies reviewed source IDs and expected answerability. Slices turn a vague regression into a diagnosis: exact lookup, paraphrase, no-answer, permission boundary, freshness, or adversarial content. Keep an independent holdout so tuning does not turn test data into training data.

In [ ]:
from examples.intermediate.evaluation import (
    EvalCase, EvalObservation, ReleaseGate, evaluate, evaluate_slices, release_report
)

cases = [
    EvalCase('checkout error 42 after 08:42 release', frozenset({'deployment-842', 'incident-eu'}), 'exact'),
    EvalCase('European checkout is slow after the release', frozenset({'deployment-842', 'incident-eu'}), 'paraphrase'),
    EvalCase('Which planet has rings?', frozenset(), 'no-answer', answerable=False),
    EvalCase('Globex private rollback procedure', frozenset(), 'authorization', answerable=False),
]
[(case.query, case.slice, case.answerable, sorted(case.relevant_ids)) for case in cases]

## 2. Evaluate retrieval separately from the answer

Recall@K asks whether all labeled evidence appeared. Precision@K measures noise. MRR rewards the first relevant result, and nDCG evaluates ordering across several labels. None proves that generated text stayed within the evidence. Record IDs after retrieval, filtering, reranking, and final context so you can locate the failing stage.

In [ ]:
retrievals = {
    'checkout error 42 after 08:42 release': ['deployment-842', 'incident-eu', 'health-guide'],
    'European checkout is slow after the release': ['incident-eu', 'deployment-842', 'support-sla'],
    'Which planet has rings?': [],
    'Globex private rollback procedure': [],
}
metrics = evaluate(retrievals, cases, k=3)
slice_metrics = evaluate_slices(retrievals, cases, k=3)
print(metrics)
print(slice_metrics)
assert metrics['recall@k'] == 1.0
assert slice_metrics['paraphrase']['mrr'] == 1.0

## 3. Add answer support, citations, and abstention

A deterministic evaluator can verify that an answer has citations and a policy-compliant response. A human or calibrated LLM judge can evaluate nuanced groundedness and helpfulness. Keep the judge separate from the system under test, version its rubric/model, and sample disagreements for human review. For a no-answer or unauthorized request, abstaining is the correct outcome—not an error.

In [ ]:
observations = [
    EvalObservation(cases[0].query, tuple(retrievals[cases[0].query]), ('deployment-842', 'incident-eu'), True, True, 760, 0.012),
    EvalObservation(cases[1].query, tuple(retrievals[cases[1].query]), ('incident-eu', 'deployment-842'), True, True, 840, 0.014),
    EvalObservation(cases[2].query, (), (), False, False, 190, 0.004),
    EvalObservation(cases[3].query, (), (), False, False, 205, 0.004),
]
report = release_report(retrievals, cases, observations, gate=ReleaseGate(minimum_mrr=0.8))
print({key: report[key] for key in ('ship', 'citation_coverage', 'abstention_accuracy', 'p95_latency_ms', 'average_cost')})
print(report['checks'])
assert report['ship']

## 4. Make a regression visible

Now simulate a configuration that returns generic health content for the paraphrase. The release report does not hide this behind a single score: it shows failed retrieval checks and the affected slice. In production, also block for unauthorized IDs, unsafe tool behavior, or citation-verification failure.

In [ ]:
bad_retrievals = {**retrievals, 'European checkout is slow after the release': ['health-guide', 'support-sla']}
bad_report = release_report(bad_retrievals, cases, observations, gate=ReleaseGate(minimum_recall=0.9, minimum_mrr=0.9))
print('ship?', bad_report['ship'])
print('failed checks:', [name for name, passed in bad_report['checks'].items() if not passed])
print('paraphrase slice:', bad_report['slice_metrics']['paraphrase'])
assert not bad_report['ship']
assert bad_report['slice_metrics']['paraphrase']['recall@k'] == 0.0

## 5. Production evaluation playbook

1. Version cases, corpus snapshot, policies, retriever/reranker, prompt, judge, and thresholds together.
2. Run deterministic retrieval/policy checks in CI; schedule judge and human-review evaluations with budgets.
3. Gate releases on hard safety/citation constraints plus measured quality and operational thresholds.
4. Canary, monitor slice-level drift and p95/cost, preserve traces, and define rollback before deployment.
5. Refresh cases from reviewed production failures while keeping a held-out set.

### Exercises

- Add a stale-source case and define whether the safe answer is historical context or abstention.
- Add an injected retrieved document and assert it cannot change policy or cite unauthorized evidence.
- Create a rubric for ‘supported mitigation recommendation’; score it with two reviewers and inspect disagreement.
- Change one retrieval parameter and compare candidate recall, final nDCG, p95 latency, and cost per successful case.

### References

- [Ragas paper](https://arxiv.org/abs/2309.15217) and [metric catalog](https://docs.ragas.io/en/stable/concepts/metrics/available_metrics/)
- [NIST Generative AI Profile](https://nvlpubs.nist.gov/nistpubs/ai/NIST.AI.600-1.pdf)
- [RAG survey](https://arxiv.org/abs/2312.10997)